# 📊 Polymarket: Datafication & the Wisdom(?) of Crowds


In [ ]:
# install dependencies, import what we need

!pip install requests pandas websocket-client --quiet

In [ ]:
import requests
import pandas as pd
import json

print('✅ libraries loaded')

---
## 1️⃣ First - what is a 'market' in this case?

We'll use the **Gamma API** — Polymarket's free public API.

Check out their 'Getting Started' documentation: https://help.polymarket.com/en/articles/13364060-what-is-polymarket

Take some notes here:

- What is a 'market,' in Polymarket's world?
- Are there things that you have questions about, from this documentation alone?

In [ ]:
# query the API

GAMMA = "https://gamma-api.polymarket.com"

# make a request here
resp = requests.get(f"{GAMMA}/markets", params={
    "active": "true",
    "closed": "false",
    "limit": 30,
    "order": "volume24hr",
    "ascending": "false"
})

# store the JSON object in markets_raw

markets_raw = resp.json()
print(f"✅ Fetched {len(markets_raw)} markets")

---
## DataFrame

Each row = one 'market'.

According to Polymarket, prices are **implied probabilities** (0 to 1). What do you think about this connection?

In [ ]:
# print your results from the API

print(markets_raw)

In [ ]:
# use this cell to put the results in a pandas dataFrame
















In [ ]:
# use this cell to find the top 10 'market' questions by trading VOLUME







In [ ]:
# use this cell to pull out a few more learnings from the markets_raw object ...
# what can you learn from this 1 data pull?






---
## 3️⃣ Live WebSocket Stream 🌊


> Take a look at the code below - how is this creating a simulation of a kind of "stream" from the API?Run the cell below. It will stream for ~60 seconds then stop automatically.  

> ⚠️  Press the **■ Stop** button in the toolbar to stop it early.

In [ ]:
###### import websocket
import threading
import time
from IPython.display import display, clear_output

# ── Grab the top 5 token IDs to subscribe to ─────────────────────────────────
token_ids = df.nlargest(5, 'volume_24h')['clob_token_id'].tolist()
token_ids = [t for t in token_ids if t != '?'][:5]

stream_events = []
STREAM_SECONDS = 60  # auto-stop after this long

def on_open(ws):
    print(f"🟢 Connected! Subscribing to {len(token_ids)} markets...")
    sub = {"assets_ids": token_ids, "type": "Market"}
    ws.send(json.dumps(sub))

def on_message(ws, message):
    try:
        events = json.loads(message)
        if not isinstance(events, list):
            events = [events]
        for event in events:
            event_type = event.get('event_type', event.get('type', 'unknown'))
            asset_id   = event.get('asset_id', event.get('market', ''))[:12] + '...'
            price      = event.get('price', event.get('last_trade_price', None))
            stream_events.append({
                'type':     event_type,
                'asset':    asset_id,
                'price':    float(price) if price else None,
                'raw':      str(event)[:80]
            })
        clear_output(wait=True)
        sdf = pd.DataFrame(stream_events[-20:])  # show last 20 events
        print(f"🔴 LIVE STREAM — {len(stream_events)} events received so far")
        print(f"   Streaming for up to {STREAM_SECONDS}s. Stop button to end early.\n")
        display(sdf)
    except Exception as e:
        pass

def on_error(ws, error):
    print(f"⚠️ WebSocket error: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f"\n🔴 Stream closed. Total events received: {len(stream_events)}")

ws = websocket.WebSocketApp(
    "wss://ws-subscriptions-clob.polymarket.com/ws/market",
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close
)

# Run in background thread, auto-close after STREAM_SECONDS
import ssl
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

t = threading.Thread(target=lambda: ws.run_forever(sslopt={"context": ssl_ctx}))

t.daemon = True
t.start()

time.sleep(STREAM_SECONDS)
ws.close()
print("\n✅ Stream ended.")

---
## 💬 Discussion Questions

### Datafication vs. Wisdom of Crowds - take some notes here

Start by reading the Introduction of this review on the concept of **datafication**: https://policyreview.info/concepts/datafication

1. **What is being datafied here?** What are we actually looking at when we look at this stream?

2. **Does money make the crowd wiser?** Prediction markets require skin in the game. Does that make them more accurate than polls or expert forecasts?

3. **Who is the crowd?** Look at the volume numbers. Is this really a "crowd" or a small number of large traders?

4. **Regulation:** Should governments or other regulators be able to shut down prediction markets on political events?